# Finding models in WebLLM

This notebook is about the **model-browsing** part of WebLLM. You will see what models are available, learn how to read their names, look up their model cards, download one, check what is saved in your browser, and delete one when you need space.

Everything runs in your browser. Nothing is sent to a server.

**Before you start:** use a recent Chrome or Edge browser on a laptop. The first download can take a while, but after that the model is saved in your browser.

## Step 1: Get set up

Small cells that bring in the Python tools we need, check that your browser can run models, load the WebLLM library, and set up the model loader.

### 1a: Import tools

- `json` turns Python data into text and back. We use it to hand data to JavaScript.
- `pyjs` connects Python to the browser's **JavaScript**, the language web pages run on. WebLLM is written in JavaScript, so we need this bridge.
- `pandas` (as `pd`) makes and filters tables.
- `re` finds patterns in text (these patterns are called **regular expressions**).
- `display` lets a cell show something and then update it in place (we use it for a progress bar).

In [ ]:
import json
import pyjs
import pandas as pd
import re
from IPython.display import display

js = pyjs.js  # a doorway from Python into the browser's JavaScript

print("Tools imported.")

### 1b: Check for WebGPU

Models run on your computer's graphics chip (GPU). Browsers reach the GPU through a feature called **WebGPU**. Recent Chrome and Edge on a laptop have it; many phones and older browsers do not.

In [ ]:
if getattr(js.navigator, "gpu", None) is not None:
    print("WebGPU is available. You can run models.")
else:
    print("WebGPU is not available here. Try the latest Chrome or Edge on a laptop.")

### 1c: Load WebLLM

**WebLLM** is a free JavaScript library that runs language models inside the browser. This line fetches it from the internet (it takes a few seconds the first time).

In [ ]:
webllm = await js.eval("import('https://esm.run/@mlc-ai/web-llm')")
print("WebLLM loaded.")

### 1d: Translate between Python and JavaScript

WebLLM runs in JavaScript, and JavaScript cannot read Python dictionaries directly. Both languages understand **JSON**, a plain-text format for data, so we translate by way of JSON text:

- `js_obj()` turns a Python dictionary into a JavaScript object, for requests we send to WebLLM.
- `py_obj()` goes the other way, turning a JavaScript object (like the model's reply) into Python dictionaries and lists.

You met `js_obj()` in Step 7c of the starter notebook.

In [ ]:
def js_obj(d):
    """Turn a Python dict into a JavaScript object (by way of JSON text)."""
    return js.JSON.parse(json.dumps(d))


def py_obj(value):
    """Turn a JavaScript object into Python dicts and lists (by way of JSON text)."""
    return json.loads(js.JSON.stringify(value))


print("Translators ready.")

### 1e: Set up the model loader

- `engine` is the part of WebLLM that actually runs a model. It starts as `None` (Python for "nothing yet") and gets created the first time we need it.
- `loaded_model` remembers the name of the model that is currently loaded, so later steps can check that one is ready.
- `progress_bar()` shows a bar that fills up while a model downloads or loads.
- `start_model()` gets a model ready to run. If the model is not saved in your browser yet, it downloads it first.

This is the same loader as in the starter notebook. You do not need to follow every line. Just run it.

In [ ]:
engine = None        # the thing that runs a model
loaded_model = None  # which model is currently loaded


def progress_bar():
    """Show a progress bar that updates in place."""
    status = display("Starting...", display_id=True)

    def show(report):
        filled = int(report.progress * 30)
        bar = "#" * filled + "-" * (30 - filled)
        status.update(f"[{bar}] {report.progress:.0%}\n{report.text}")

    # JavaScript needs its own handle on the Python function to call it
    js_show, handle = pyjs.create_callable(show)
    return status, js_show, handle


async def start_model(model_id):
    """Get a model ready to run. Downloads it only if it is not already saved."""
    global engine
    status, progress, handle = progress_bar()

    if engine is None:
        engine = webllm.MLCEngine.new()   # create an empty engine
    engine.setInitProgressCallback(progress)

    try:
        await engine.reload(model_id)
    except Exception as error:
        # Some models have two conflicting memory-window settings.
        # If so, turn off the "sliding window" one and try again.
        if "WindowSizeConfigurationError" in str(error):
            status.update("Adjusting this model's settings and retrying...")
            await engine.reload(model_id, js_obj({"sliding_window_size": -1}))
        else:
            raise

print("Model loader is ready.")

## Step 2: See which models are available

WebLLM comes with a long list of ready-to-use models. We will turn that list into a table, add a few useful columns, and then filter it down to the models worth trying. These are the same steps as in the starter notebook.

### 2a: Look at WebLLM's model list

WebLLM keeps its list in `webllm.prebuiltAppConfig.model_list`. Each entry describes one model. Let's count them and look at the first one.

In [ ]:
model_list = list(webllm.prebuiltAppConfig.model_list)
print(f"WebLLM knows about {len(model_list)} models.\n")

first = model_list[0]
print("model_id:        ", first.model_id)
print("vram_required_MB:", first.vram_required_MB)

### 2b: Turn the list into a table

We keep three things about each model:

- **`name`**: the model's ID, which we use to download and load it.
- **`size_mb`**: how much graphics memory it needs, in megabytes (MB). 1000 MB is about 1 GB.
- **`type`**: what kind of model it is. WebLLM stores this as a number, so `TYPES` turns the number into a word:
  - `"chat"` models answer questions. This is what we use in this notebook.
  - `"embedding"` models turn text into lists of numbers, used for search. We will use these later.
  - `"vision"` models can look at images.

In [ ]:
TYPES = {0: "chat", 1: "embedding", 2: "vision"}

rows = []
for m in model_list:
    size = getattr(m, "vram_required_MB", None)
    rows.append({
        "name": m.model_id,
        "size_mb": round(size) if size else None,
        "type": TYPES.get(getattr(m, "model_type", None) or 0, "other"),
    })

table = pd.DataFrame(rows)
table["size_mb"] = table["size_mb"].astype("Int64")  # whole numbers, blank if unknown
table.head()

### 2c: Add release dates

WebLLM does not record when each model came out, so we keep our own small table, `RELEASED`, with the month each model **family** was first released (`"YYYY-MM"`). A model gets the date of the first family its name starts with.

These dates are approximate and typed in by hand, so this table **will fall behind** as new models are added to WebLLM. New models show up with a `?`. To fix one, add a line like `("NewFamily-", "2026-05"),` near the top. Put more specific names before less specific ones (`Qwen3.5-` before `Qwen3-`), because the first match wins.

In [ ]:
RELEASED = [
    ("Qwen3.5-", "2026-03"),
    ("Ministral-3-", "2025-12"),
    ("Qwen3-", "2025-04"),
    ("OLMo-2-0425", "2025-04"),
    ("gemma3-", "2025-03"),
    ("Phi-4-mini", "2025-02"),
    ("DeepSeek-R1-Distill", "2025-01"),
    ("SmolLM2-", "2024-11"),
    ("OLMo-2-1124", "2024-11"),
    ("Qwen2.5-Coder", "2024-11"),
    ("Qwen2.5-", "2024-09"),
    ("Llama-3.2-", "2024-09"),
    ("Phi-3.5", "2024-08"),
    ("Hermes-3", "2024-08"),
    ("gemma-2-", "2024-07"),
    ("Llama-3.1-", "2024-07"),
    ("Qwen2-", "2024-06"),
    ("Mistral-7B-Instruct-v0.3", "2024-05"),
    ("Llama-3-", "2024-04"),
    ("Phi-3-", "2024-04"),
    ("snowflake-arctic-embed", "2024-04"),
    ("Hermes-2-Theta", "2024-05"),
    ("Hermes-2-Pro", "2024-03"),
    ("gemma-2b-it", "2024-02"),
    ("TinyLlama", "2024-01"),
    ("stablelm-2-zephyr", "2024-01"),
    ("phi-2", "2023-12"),
    ("Mistral-7B-Instruct-v0.2", "2023-12"),
    ("NeuralHermes-2.5", "2023-12"),
    ("WizardMath-7B-V1.1", "2023-12"),
    ("OpenHermes-2.5", "2023-11"),
    ("phi-1_5", "2023-09"),
    ("Llama-2", "2023-07"),
    ("RedPajama", "2023-05"),
]


def released(name):
    for prefix, month in RELEASED:
        if name.lower().startswith(prefix.lower()):
            return month
    return "?"


table["released"] = table["name"].apply(released)
table.head()

### 2d: Mark which models are already saved

Once you download a model, your browser keeps it. `webllm.hasModelInCache(name)` checks whether a model is saved. We ask about every model and store the answer in a `saved` column.

In [ ]:
saved = []
for name in table["name"]:
    saved.append(await webllm.hasModelInCache(name))

table["saved"] = saved
print(f"{sum(saved)} of {len(table)} models are saved in your browser.")

### 2e: Filter the table

Now narrow the table down. Change the settings at the top of the cell and run it again:

- **`MODEL_TYPE`**: which kind of model to show (`"chat"`, `"embedding"` or `"vision"`).
- **`MAX_SIZE_MB`**: only show models that need at most this much graphics memory. **Start with 2000** (about 2 GB). Bigger models give better answers but download slowly and can be painfully slow to run on a laptop. If 2 GB models run well on your computer, try 4000.
- **`NEWER_THAN`**: only show models released in or after this month. Models with a `?` date are kept, since we do not know when they came out.

Each model also comes in a few versions. We only keep the **`q4f16`** version, which is compressed to about a quarter of the original size and is the best choice for laptops.

Each filter line keeps only the rows where the condition is true. The list is sorted newest first. To pick a model in Step 3, copy its `name`.

In [ ]:
MODEL_TYPE = "chat"
MAX_SIZE_MB = 2000
NEWER_THAN = "2025-01"

models = table
models = models[models["type"] == MODEL_TYPE]
models = models[models["name"].str.contains("q4f16")]
models = models[models["size_mb"] <= MAX_SIZE_MB]
models = models[(models["released"] >= NEWER_THAN) | (models["released"] == "?")]

# Newest first. Treat "?" as oldest so unknown dates go to the bottom.
models = models.sort_values(["released", "size_mb"], ascending=False,
                            key=lambda column: column.replace("?", ""))

print(f"{len(models)} models match your filters (newest first)")
models

### Optional: search the full model list

The table in 2e only shows models that pass your filters. To look up **any** model WebLLM supports, put a word in `SEARCH` (like `"qwen"`, `"phi"` or `"smol"`). Set it to `""` to see every model at once (it is a long list).

In [ ]:
SEARCH = "gemma"

everything = table.sort_values(["released", "size_mb"], ascending=[False, True])
if SEARCH:
    everything = everything[everything["name"].str.contains(SEARCH, case=False)]

pd.set_option("display.max_rows", None)
print(f"{len(everything)} models")
everything

## Step 3: Learn how to read a model name

Model names look complicated at first, but they usually contain a few useful clues:

- the **family** name, like Gemma, Llama, Qwen, or SmolLM
- the **size**, like `1B` for about one billion **parameters** or `500M` for about five hundred million. A parameter is one number the model learned during training; more parameters usually means a more capable but slower model.
- a word like **`Instruct`** (or `it`), which means the model was tuned for question-answering and chat
- the **compression** tag, like **`q4f16`**, which tells you how the model was packed to fit on a smaller machine

The next cell uses a few simple rules (regular expressions, from `re`) to pull those pieces out of `EXAMPLE_MODEL`. It is not perfect, but it helps you see the pattern. Paste in any name from the tables above.

In [ ]:
EXAMPLE_MODEL = "gemma3-1b-it-q4f16_1-MLC"

size_match = re.search(r"(\d+(?:\.\d+)?)([BbMm])", EXAMPLE_MODEL)
compression_match = re.search(r"(q\d+f\d+(?:_\d+)?)", EXAMPLE_MODEL, flags=re.IGNORECASE)

family = EXAMPLE_MODEL
if size_match:
    family = EXAMPLE_MODEL[:size_match.start()].rstrip("-_")

if size_match:
    number, unit = size_match.groups()
    if unit.upper() == "B":
        size_text = f"about {number} billion parameters"
    else:
        size_text = f"about {number} million parameters"
else:
    size_text = "no size pattern found"

if "instruct" in EXAMPLE_MODEL.lower() or "-it-" in EXAMPLE_MODEL.lower():
    tuning_text = "This looks like an instruction-tuned model for chat or question-answering."
else:
    tuning_text = "This name does not clearly say it is instruction-tuned."

print(f"Model name: {EXAMPLE_MODEL}\n")
print(f"Family: {family}")
print(f"Size: {size_text}")
print(f"Compression tag: {compression_match.group(1) if compression_match else 'not found'}")
print(tuning_text)
print("\nTip: q4f16 is a compressed version that is much smaller than the original model, which is why it can fit in a browser.")

## Step 4: Look up the model card on Hugging Face

**Hugging Face** is a website where people share models. A **model card** is a model's information page there. It usually tells you:

- who made the model
- what data or training recipe was used
- the license
- the intended use
- the limitations, risks, or warnings

For WebLLM models, there are often **two** useful Hugging Face pages:

1. the **original model** from the model maker
2. the **converted model** from `mlc-ai`, which is the version prepared for WebLLM

The next cell prints both links for `MODEL_FOR_CARD`. `MODEL_CARDS` is a small hand-made table, like `RELEASED`. For a model that is not in it, the original model's page has to be found by searching Hugging Face.

In [ ]:
MODEL_FOR_CARD = "gemma3-1b-it-q4f16_1-MLC"

MODEL_CARDS = {
    "gemma3-1b-it-q4f16_1-MLC": "google/gemma-3-1b-it",
    "Llama-3.2-1B-Instruct-q4f16_1-MLC": "meta-llama/Llama-3.2-1B-Instruct",
    "Qwen2.5-0.5B-Instruct-q4f16_1-MLC": "Qwen/Qwen2.5-0.5B-Instruct",
    "SmolLM2-360M-Instruct-q4f16_1-MLC": "HuggingFaceTB/SmolLM2-360M-Instruct",
    "snowflake-arctic-embed-m-q0f32-MLC": "Snowflake/snowflake-arctic-embed-m",
}

original_repo = MODEL_CARDS.get(MODEL_FOR_CARD)

print(f"WebLLM model: {MODEL_FOR_CARD}\n")
if original_repo:
    print(f"Original model card:  https://huggingface.co/{original_repo}")
else:
    print("Original model card:  not in MODEL_CARDS yet. Search for the name on https://huggingface.co")
print(f"Converted model card: https://huggingface.co/mlc-ai/{MODEL_FOR_CARD}")

### 4a: Who makes some of the common small models?

When you browse models, do not just look at size. Also ask:

- **Who made it?**
- **What does the model card say about training data or source materials?**
- **What license is it under?**
- **What limits or warnings does the model card mention?**

Here are six small-model families you are likely to see:

- **Gemma 3** is made by **Google**. Start with the model card at <https://huggingface.co/google/gemma-3-1b-it>. On Hugging Face, you may need to click through the Gemma license before you can read everything. This is a good place to look for Google's description of intended use, license terms, and the high-level training story.
- **Llama 3.2** is made by **Meta**. Start with the model card at <https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct> and the Llama site at <https://www.llama.com/>. On Hugging Face, you may need to accept the Llama license. Look here for supported languages, usage terms, and the official description of the training setup.
- **Qwen2.5** is made by the **Qwen team at Alibaba Cloud**. Start with the model card at <https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct>, the project blog at <https://qwenlm.github.io/blog/qwen2.5/>, and the docs at <https://qwen.readthedocs.io/>. The model card says this family supports many languages and gives a clear summary of its design, parameter count, and context length.
- **SmolLM2** is made by **Hugging Face**. Start with the model card at <https://huggingface.co/HuggingFaceTB/SmolLM2-360M-Instruct> and the project repo at <https://github.com/huggingface/smollm>. The model card says the 360M model was trained on 4 trillion tokens using datasets such as FineWeb-Edu, DCLM, and The Stack. This is a good family for students who want a detailed public story about data and training.
- **OLMo 2** is made by **Ai2, the Allen Institute for AI**. Start with the model card at <https://huggingface.co/allenai/OLMo-2-0425-1B-Instruct> and the project page at <https://allenai.org/olmo>. OLMo is designed as an open research model family, trained on the Dolma dataset, with code, checkpoints, logs, and training details released publicly.
- **Phi** is made by **Microsoft**. Start with the model card at <https://huggingface.co/microsoft/Phi-3.5-mini-instruct>. The model card says Phi-3.5 mini is built from synthetic data plus filtered public websites, with a focus on high-quality reasoning data.

A good habit is to open both pages (original and `mlc-ai`) for any WebLLM model you want to study, then scan for four things: **who made it, what it was trained on, what the license allows, and what the limitations are.**

## Step 5: Pick a model to download

Put the **name** of a model in `DOWNLOAD_NAME`. Copy it exactly from a table above. The default, **Gemma 3 1B** from Google, is small, quick and gives good answers, so it is a good first model.

In [ ]:
DOWNLOAD_NAME = "gemma3-1b-it-q4f16_1-MLC"

pick = table[table["name"] == DOWNLOAD_NAME]

if len(pick) == 0:
    print(f"No model is called {DOWNLOAD_NAME}. Check the spelling against the table.")
else:
    print(f"You picked: {DOWNLOAD_NAME} (about {pick['size_mb'].iloc[0]} MB)")

## Step 6: Download it

The `if` checks whether the model is already saved in your browser. If it is, there is nothing to do. If not (`else`), `start_model()` downloads it.

`start_model()` also loads the model onto your graphics chip, which takes up memory. Since we only wanted to download for now, `engine.unload()` frees that memory again. To chat with a model, see the starter notebook or notebook 02.

In [ ]:
if await webllm.hasModelInCache(DOWNLOAD_NAME):
    print(f"{DOWNLOAD_NAME} is already saved. Nothing to download.")
else:
    await start_model(DOWNLOAD_NAME)
    await engine.unload()   # downloaded and saved; free up memory for now
    loaded_model = None
    print(f"{DOWNLOAD_NAME} is downloaded and saved in your browser.")

## Step 7: Check what you have saved

This checks again which models are saved in your browser (you may have just downloaded one), updates the `saved` column, and shows only the saved rows.

In [ ]:
saved = []
for name in table["name"]:
    saved.append(await webllm.hasModelInCache(name))
table["saved"] = saved

saved_models = table[table["saved"]]
print(f"{len(saved_models)} models saved, about {saved_models['size_mb'].sum()} MB in total")
saved_models

**Optional: free up space.** Put the name of a saved model (from the list just above) in `DELETE_NAME` and run the cell to remove it. Leave it as `""` to do nothing.

In [ ]:
DELETE_NAME = ""

if DELETE_NAME == "":
    print("Nothing deleted.")
elif not await webllm.hasModelInCache(DELETE_NAME):
    print(f"{DELETE_NAME} is not saved, so there is nothing to delete.")
else:
    await webllm.deleteModelAllInfoInCache(DELETE_NAME)
    print(f"Deleted {DELETE_NAME}. Run Step 7 again to see the updated list.")

## Step 8: Other places to browse models

Two other places are useful when you want to explore beyond this notebook:

- WebLLM's model list in `src/config.ts`: <https://github.com/mlc-ai/web-llm/blob/main/src/config.ts>
- The WebLLM Chat demo: <https://chat.webllm.ai/>

The notebook is easier for learning, but those two places are handy when you want to see what is new.